In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr libtesseract-dev
!pip install pytesseract flask


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,769 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get

In [ ]:
!wget -q -nc https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip
!unzip -o ngrok-v3-stable-linux-amd64.zip
!./ngrok config add-authtoken 2vfpTy4d5s3wucxdcHlvzbFDsD4_XfbNT9JBRDaJwYrNfSBy


Archive:  ngrok-v3-stable-linux-amd64.zip
  inflating: ngrok                   
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import subprocess
import time
import requests

ngrok = subprocess.Popen(['./ngrok', 'http', '5000'])
time.sleep(4)

try:
    r = requests.get("http://localhost:4040/api/tunnels")
    public_url = r.json()['tunnels'][0]['public_url']
    print("Public ngrok URL:", public_url)
except Exception as e:
    print("Error fetching ngrok URL:", e)


Public ngrok URL: https://c751-34-125-244-13.ngrok-free.app


In [ ]:
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r'/usr/bin/tesseract'


In [ ]:
from flask import Flask, request, jsonify
from PIL import Image
import cv2
import pytesseract
import re
import os
import logging

app = Flask(__name__)

logging.basicConfig(level=logging.INFO)

@app.route('/')
def home():
    return """ <html>
    <head>
        <title>Aadhaar Extractor</title>
        <style>
            body {font-family: Arial, sans-serif; display: flex; justify-content: center; align-items: center; height: 100vh; background-color: #f4f4f4;}
            .container {background: white; padding: 20px; border-radius: 10px; box-shadow: 0 0 10px rgba(0, 0, 0, 0.1); width: 300px; text-align: center;}
            input[type="file"], input[type="text"] {width: 100%; padding: 10px; margin: 10px 0; border: 1px solid #ccc; border-radius: 5px;}
            button {background: #007bff; color: white; border: none; padding: 10px; width: 100%; border-radius: 5px; cursor: pointer;}
            button:disabled {background: #ccc;}
        </style>
    </head>
    <body>
        <div class="container">
            <h2>Upload Aadhaar Card</h2>
            <input type="file" id="fileInput" accept="image/*">
            <button onclick="uploadFile()">Upload Aadhaar Card</button>
            <input type="text" id="name" placeholder="Name">
            <input type="text" id="dob" placeholder="Date of Birth">
            <input type="text" id="aadhaar" placeholder="Aadhaar Number">
            <input type="text" id="gender" placeholder="Gender">
        </div>
        <script>
            async function uploadFile() {
                let fileInput = document.getElementById("fileInput");
                let button = document.querySelector("button");

                if (fileInput.files.length === 0) {
                    alert("Please select a file");
                    return;
                }

                let formData = new FormData();
                formData.append("file", fileInput.files[0]);

                button.disabled = true;
                button.innerText = "Uploading...";

                try {
                    let response = await fetch("/upload", { method: "POST", body: formData });
                    let data = await response.json();

                    if (data.error) {
                        alert(data.error);
                    } else {
                        document.getElementById("name").value = data.name;
                        document.getElementById("dob").value = data.dob;
                        document.getElementById("aadhaar").value = data.aadhaar;
                        document.getElementById("gender").value = data.gender;
                    }

                } catch (error) {
                    console.error("Error uploading file:", error);
                    alert("Failed to upload file.");
                } finally {
                    button.disabled = false;
                    button.innerText = "Upload Aadhaar Card";
                }
            }
        </script>
    </body>
    </html>
    """

@app.route('/upload', methods=['POST'])
def upload():
    if 'file' not in request.files:
        return jsonify({'error': 'No file uploaded!'}), 400

    file = request.files['file']
    filepath = os.path.join('/tmp', file.filename)

    try:
        file.save(filepath)


        image = cv2.imread(filepath)
        if image is None:
            return jsonify({'error': 'Invalid image file.'}), 400

        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        _, threshold_image = cv2.threshold(gray_image, 150, 255, cv2.THRESH_BINARY)


        text = pytesseract.image_to_string(threshold_image)
        raw_text = pytesseract.image_to_string(Image.open(filepath))
        extracted_text = (text + raw_text).replace("®", "").strip()

        logging.info(f"Extracted Text: {extracted_text}")


        name_match = re.search(r'([A-Za-z ]+)\s*Date', extracted_text, re.MULTILINE)
        name = name_match.group(1).strip() if name_match else "Not Found"


        dob_match = re.search(r'(\d{2}[-/|]\d{2}[-/|]\d{4})', extracted_text)
        dob = dob_match.group(1).replace('|', '/').replace('-', '/') if dob_match else "Not Found"


        aadhaar_match = re.search(r'\b\d{4}\s?\d{4}\s?\d{4}\b', extracted_text)
        aadhaar = aadhaar_match.group(0).replace(' ', '') if aadhaar_match else "Not Found"


        gender_match = re.search(r'\b(MALE|FEMALE|MAL[EI]|FEMAL[EI])\b', extracted_text, re.IGNORECASE)
        if gender_match:
            gender = gender_match.group(1).upper().replace('I', 'E')
            if gender not in ['MALE', 'FEMALE']:
                gender = "Not Found"
        else:
            gender = "Not Found"

        return jsonify({'name': name, 'dob': dob, 'aadhaar': aadhaar, 'gender': gender})

    except Exception as e:
        logging.error(f"Error processing file: {str(e)}")
        return jsonify({'error': 'Internal server error.'}), 500

    finally:

        if os.path.exists(filepath):
            os.remove(filepath)

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [21/Feb/2026 03:44:59] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [21/Feb/2026 03:44:59] "GET /favicon.ico HTTP/1.1" 404 -
